## Functionality of this script
   1. Downloads NDBC spectral wave density data
   2. Computes Hm0, Te, Tp, J, and Tz using MHKiT
   3. Clusters the sea states into X representative wave conditions
   4. Saves precomputed `.mat` and `.csv` files for WEC-Sim/MATLAB
## For WEC-Sim damping optimization, the most important saved variables are from this framework are:
 -  condition_id
 -   H = Hm0
 -   T = Tp
 -   Hm0
 -   Te
 -   Tp
 -   weights
 -   probability
For full documentation please see the MHKIT PacWave Assesment here:

https://mhkit-software.github.io/MHKiT/PacWave_resource_characterization_example.html

Imported package dependencies needed for this script.

In [ ]:
from pathlib import Path
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.mixture import GaussianMixture
from scipy.io import savemat

from mhkit.wave import resource
from mhkit.wave.io import ndbc

User inputs

In [ ]:
# Change this to your project location
PROJECT_ROOT = Path.home() / "WEC-Sim_Applications" / "OSWEC_Optimization_Damping"

# NDBC buoy number
buoy_number = "46050"

# Years to analyze
years = ["2020", "2021", "2022", "2023", "2024", "2025"]

# Number of representative wave conditions to create.
# These are your X values.
clusters = [4, 8, 16, 32, 64]

# Water depth in meters
water_depth = 160.0

# Choose which cluster result to export.
# Set to a number like 32 to export only one file.
# Set to None to export all cluster cases listed above.
cluster_to_export = None

# Make diagnostic cluster plots?
# Keep False for clean production runs.
make_plots = False

# Random seed for reproducible clustering
random_state = 1


Project folder set up

In [ ]:
os.chdir(PROJECT_ROOT)

output_folder = PROJECT_ROOT / "wave_conditions"
output_folder.mkdir(parents=True, exist_ok=True)

print("--------------------------------------------")
print("Current working directory:")
print(Path.cwd())
print("--------------------------------------------")
print(f"Project root: {PROJECT_ROOT}")
print(f"Output folder: {output_folder}")
print("--------------------------------------------")

Download the spectral wave density data from the NDBC

In [ ]:
parameter = "swden"

print(f"Buoy number: {buoy_number}")
print(f"Years: {years}")
print(f"Clusters: {clusters}")
print(f"Water depth: {water_depth} m")
print("--------------------------------------------")

print("Checking available NDBC data...")

ndbc_available_data = ndbc.available_data(parameter, buoy_number)

# Create a clean year string for filenames
if years is None:
    years_string = "all_years"
else:
    years = [str(year) for year in years]
    years_string = "_".join(years)

    # Filter available files by selected years
    year_pattern = "|".join(years)

    ndbc_available_data = ndbc_available_data[
        ndbc_available_data["filename"].astype(str).str.contains(year_pattern)
    ]

    if ndbc_available_data.empty:
        raise ValueError(
            f"No data files found for buoy {buoy_number} and years {years}."
        )

filenames = ndbc_available_data["filename"]

print("Downloading NDBC data...")

ndbc_requested_data = ndbc.request_data(parameter, filenames)

Clean NDBC dara and create datetime index

In [ ]:
ndbc_data = {}

for year in ndbc_requested_data:
    print(f"Cleaning data for year {year}...")

    data_raw = ndbc_requested_data[year].copy()

    # Create datetime index from NDBC date/time columns
    data_raw["date"] = pd.to_datetime(
        {
            "year": data_raw["#YY"],
            "month": data_raw["MM"],
            "day": data_raw["DD"],
            "hour": data_raw["hh"],
            "minute": data_raw["mm"],
        },
        errors="coerce",
    )

    data_raw = data_raw.set_index("date")

    # Drop original date/time columns
    data_raw = data_raw.drop(columns=["#YY", "MM", "DD", "hh", "mm"])

    # Convert frequency column names to floats
    new_columns = []

    for col in data_raw.columns:
        col_string = str(col)

        if col_string.startswith("."):
            col_string = "0" + col_string

        new_columns.append(float(col_string))

    data_raw.columns = new_columns

    # Convert data values to numeric
    data_raw = data_raw.apply(pd.to_numeric, errors="coerce")

    # Replace NDBC missing/bad values
    data_raw = data_raw.replace([999.0, 99.0], np.nan)

    # Drop rows with missing data
    data_raw = data_raw.dropna()

    # Sort by time
    data_raw = data_raw.sort_index()

    ndbc_data[str(year)] = data_raw

print("Cleaned data years:")
print(list(ndbc_data.keys()))

Calculate Quantities of Interest (QOI's)

In [ ]:
Hm0_list = []
Te_list = []
J_list = []
Tp_list = []
Tz_list = []

for year in ndbc_data:
    print(f"Calculating QoIs for year {year}...")

    data_raw = ndbc_data[year]

    year_data = data_raw[data_raw != 999.0].dropna()

    # MHKiT expects frequency as rows and timestamps as columns
    spectrum = year_data.T

    Hm0_list.append(resource.significant_wave_height(spectrum))
    Te_list.append(resource.energy_period(spectrum))
    J_list.append(resource.energy_flux(spectrum, h=water_depth))

    # Peak period calculation from spectral peak frequency
    fp = spectrum.idxmax(axis=0).astype(float)
    Tp = 1.0 / fp
    Tp = pd.DataFrame(Tp, index=spectrum.columns, columns=["Tp"])
    Tp_list.append(Tp)

    Tz_list.append(resource.average_zero_crossing_period(spectrum))


Te = pd.concat(Te_list, axis=0)
Tp = pd.concat(Tp_list, axis=0)
Hm0 = pd.concat(Hm0_list, axis=0)
J = pd.concat(J_list, axis=0)
Tz = pd.concat(Tz_list, axis=0)

# Combine into one DataFrame
data = pd.concat([Hm0, Te, Tp, J, Tz], axis=1)

# Make sure columns are named correctly
data.columns = ["Hm0", "Te", "Tp", "J", "Tz"]

# Calculate mean wave steepness
data["Sm"] = data.Hm0 / (9.81 / (2.0 * np.pi) * data.Tz**2)

# Drop NaNs and sort
data.dropna(inplace=True)
data.sort_index(inplace=True)

print("--------------------------------------------")
print("QoI data preview:")
print(data.head())
print("--------------------------------------------")


Clean up 

In [ ]:
data_clean = data.copy()

# Remove unrealistic wave heights
data_clean = data_clean[data_clean.Hm0 < 20]

# Keep your original cleaning approach
sigma = data_clean.J.std()
data_clean = data_clean[data_clean.J > (data_clean.J.mean() - 0.9 * sigma)]

print(f"Number of sea states after cleaning: {len(data_clean)}")
print("--------------------------------------------")


Pacwave style Clustering Using Gaussian mixture model

In [ ]:
# Cluster on energy period and significant wave height
X = np.vstack((data_clean.Te.values, data_clean.Hm0.values)).T

results = {}

if make_plots:
    fig, axs = plt.subplots(
        len(clusters),
        1,
        figsize=(8, 5 * len(clusters)),
        sharex=True,
    )

    if len(clusters) == 1:
        axs = [axs]


for plot_index, cluster in enumerate(clusters):
    print(f"Creating {cluster} representative wave conditions...")

    gmm = GaussianMixture(
        n_components=cluster,
        random_state=random_state,
    ).fit(X)

    labels = gmm.predict(X)

    # Save cluster centers and weights
    result = pd.DataFrame(gmm.means_, columns=["Te", "Hm0"])
    result["weights"] = gmm.weights_

    # Normalize weights just to be safe
    result["weights"] = result["weights"] / result["weights"].sum()

    # Use the same relationship as the PacWave example
    result["Tp"] = result.Te / 0.858

    # Sort from smaller waves to larger waves for easier interpretation
    result = result.sort_values(["Hm0", "Te"]).reset_index(drop=True)

    # Add condition IDs after sorting
    result.insert(0, "condition_id", np.arange(1, len(result) + 1))

    # Add WEC-Sim-friendly aliases
    result["H"] = result["Hm0"]
    result["T"] = result["Tp"]

    # Use probability as clearer name for optimization weighting
    result["probability"] = result["weights"]

    results[cluster] = result

    if make_plots:
        axs[plot_index].scatter(
            data_clean.Te.values,
            data_clean.Hm0.values,
            c=labels,
            s=8,
        )

        axs[plot_index].plot(
            result.Te,
            result.Hm0,
            "m+",
            markersize=10,
        )

        axs[plot_index].set_title(f"{cluster} Clusters")
        axs[plot_index].set_ylabel("Hm0 [m]")

if make_plots:
    axs[-1].set_xlabel("Energy Period, Te [s]")
    plt.tight_layout()
    plt.show()

View selected wave conditions

In [ ]:
if cluster_to_export is not None:
    print("--------------------------------------------")
    print(f"Representative wave conditions for {cluster_to_export} clusters:")
    print("--------------------------------------------")
    print(results[cluster_to_export])

Save the representative wave conditions as `.mat` and `.csv` file types 

In [ ]:
def make_matlab_mcr_struct(result):
    """
    Create a MATLAB-compatible struct array named mcr.

    Each mcr(i) contains one wave condition:
        mcr(i).caseNum
        mcr(i).condition_id
        mcr(i).H
        mcr(i).T
        mcr(i).Hm0
        mcr(i).Te
        mcr(i).Tp
        mcr(i).weights
        mcr(i).probability
        mcr(i).caseName
    """

    n_cases = len(result)

    mcr_dtype = np.dtype(
        [
            ("caseNum", "O"),
            ("condition_id", "O"),
            ("H", "O"),
            ("T", "O"),
            ("Hm0", "O"),
            ("Te", "O"),
            ("Tp", "O"),
            ("weights", "O"),
            ("probability", "O"),
            ("caseName", "O"),
        ]
    )

    # 1-by-n MATLAB struct array
    mcr = np.empty((1, n_cases), dtype=mcr_dtype)

    for i in range(n_cases):
        mcr["caseNum"][0, i] = i + 1
        mcr["condition_id"][0, i] = float(result["condition_id"].iloc[i])

        # WEC-Sim wave inputs
        mcr["H"][0, i] = float(result["H"].iloc[i])
        mcr["T"][0, i] = float(result["T"].iloc[i])

        # Resource variables
        mcr["Hm0"][0, i] = float(result["Hm0"].iloc[i])
        mcr["Te"][0, i] = float(result["Te"].iloc[i])
        mcr["Tp"][0, i] = float(result["Tp"].iloc[i])

        # Occurrence weighting
        mcr["weights"][0, i] = float(result["weights"].iloc[i])
        mcr["probability"][0, i] = float(result["probability"].iloc[i])

        # Human-readable case name
        mcr["caseName"][0, i] = f"condition_{int(result['condition_id'].iloc[i]):03d}"

    return mcr


# Decide which cluster counts to export
if cluster_to_export is None:
    clusters_to_save = clusters
else:
    clusters_to_save = [cluster_to_export]

for cluster in clusters_to_save:
    result = results[cluster]

    # MATLAB/WEC-Sim-friendly array output
    condition_id = result["condition_id"].to_numpy(dtype=float)

    Hm0_array = result["Hm0"].to_numpy(dtype=float)
    Te_array = result["Te"].to_numpy(dtype=float)
    Tp_array = result["Tp"].to_numpy(dtype=float)

    weights_array = result["weights"].to_numpy(dtype=float)
    probability_array = result["probability"].to_numpy(dtype=float)

    # WEC-Sim aliases
    H_array = result["H"].to_numpy(dtype=float)
    T_array = result["T"].to_numpy(dtype=float)

    # Plain array-style MAT file
    wave_conditions = {
        "condition_id": condition_id,
        "Hm0": Hm0_array,
        "Te": Te_array,
        "Tp": Tp_array,
        "H": H_array,
        "T": T_array,
        "weights": weights_array,
        "probability": probability_array,
        "n_conditions": np.array([cluster], dtype=float),
        "cluster_to_export": np.array([cluster], dtype=float),
        "buoy_number": buoy_number,
        "years_string": years_string,
        "water_depth": np.array([water_depth], dtype=float),
    }

    # MCR-style MAT file
    mcr = make_matlab_mcr_struct(result)

    output_base_name = (
        f"wave_conditions_buoy_{buoy_number}_{years_string}_{cluster}_clusters"
    )

    output_mat_file = output_folder / f"{output_base_name}.mat"
    output_mcr_file = output_folder / f"{output_base_name}_mcr.mat"
    output_csv_file = output_folder / f"{output_base_name}.csv"

    # Save normal MAT file
    savemat(str(output_mat_file), wave_conditions)

    # Save WEC-Sim MCR MAT file
    savemat(
        str(output_mcr_file),
        {
            "mcr": mcr,
            "condition_id": condition_id,
            "H": H_array,
            "T": T_array,
            "Hm0": Hm0_array,
            "Te": Te_array,
            "Tp": Tp_array,
            "weights": weights_array,
            "probability": probability_array,
            "n_conditions": np.array([cluster], dtype=float),
            "cluster_to_export": np.array([cluster], dtype=float),
            "buoy_number": buoy_number,
            "years_string": years_string,
            "water_depth": np.array([water_depth], dtype=float),
        },
    )

    # Save CSV for easy inspection
    result.to_csv(output_csv_file, index=False)

    print("--------------------------------------------")
    print(f"Saved {cluster}-condition WEC-Sim wave files:")
    print(f"Array MAT: {output_mat_file}")
    print(f"MCR MAT:   {output_mcr_file}")
    print(f"CSV:       {output_csv_file}")
    print("--------------------------------------------")

print("Done.")